### Objective:

##### Transform raw YouTube trending data from the Bronze layer into a clean, analytics-ready Silver layer by performing data validation, duplicate removal, null handling, timestamp standardization, feature engineering, and KPI calculations such as Engagement Rate and Like Percentage.

#### Step1: Read Bronze Data

In [0]:
bronze_path = "/Volumes/workspace/default/youtube_data/bronze"

bronze_df = spark.read \
.format("delta") \
.load(bronze_path)

display(bronze_df.limit(10))

video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
Jw1Y-zhQURU,17.14.11,John Lewis Christmas Ad 2017 - #MozTheMonster,John Lewis,26,2017-11-10T07:38:29.000Z,"""christmas""|""john lewis christmas""|""john lewis""|""christmas ad""|""mozthemonster""|""christmas 2017""|""christmas ad 2017""|""john lewis christmas advert""|""moz""",7224515,55681,10247,9479,https://i.ytimg.com/vi/Jw1Y-zhQURU/default.jpg,False,False,False,"Click here to continue the story and make your own monster:\nhttp://bit.ly/2mboXgj\n\nJoe befriends a noisy Monster under his bed but the two have so much fun together that he can't get to sleep, leaving him tired by day. For Christmas Joe receives a gift to help him finally get a good night’s sleep.\n\nShop the ad\nhttp://bit.ly/2hg04Lc\n\nThe music is Golden Slumbers performed by elbow, the original song was by The Beatles. \nFind the track:\nhttps://Elbow.lnk.to/GoldenSlumbersXS\n\nSubscribe to this channel for regular video updates\nhttp://bit.ly/2eU8MvW\n\nIf you want to hear more from John Lewis:\n\nLike John Lewis on Facebook\nhttp://www.facebook.com/johnlewisretail\n\nFollow John Lewis on Twitter\nhttp://twitter.com/johnlewisretail\n\nFollow John Lewis on Instagram\nhttp://instagram.com/johnlewisretail",GB
3s1rvMFUweQ,17.14.11,Taylor Swift: …Ready for It? (Live) - SNL,Saturday Night Live,24,2017-11-12T06:24:44.000Z,"""SNL""|""Saturday Night Live""|""SNL Season 43""|""Episode 1730""|""Tiffany Haddish""|""Taylor Swift""|""Taylor Swift Ready for It""|""s43""|""s43e5""|""episode 5""|""live""|""new york""|""comedy""|""sketch""|""funny""|""hilarious""|""late night""|""host""|""music""|""guest""|""laugh""|""impersonation""|""actor""|""improv""|""musician""|""comedian""|""actress""|""If Loving You Is Wrong""|""Oprah Winfrey""|""OWN""|""Girls Trip""|""The Carmichael Show""|""Keanu""|""Reputation""|""Look What You Made Me Do""|""ready for it?""",1053632,25561,2294,2757,https://i.ytimg.com/vi/3s1rvMFUweQ/default.jpg,False,False,False,Musical guest Taylor Swift performs …Ready for It? on Saturday Night Live.\n\n#SNL #SNL43\n\nGet more SNL: http://www.nbc.com/saturday-night-live\nFull Episodes: http://www.nbc.com/saturday-night-liv...\n\nLike SNL: https://www.facebook.com/snl\nFollow SNL: https://twitter.com/nbcsnl\nSNL Tumblr: http://nbcsnl.tumblr.com/\nSNL Instagram: http://instagram.com/nbcsnl \nSNL Pinterest: http://www.pinterest.com/nbcsnl/,GB
n1WpP7iowLc,17.14.11,Eminem - Walk On Water (Audio) ft. Beyoncé,EminemVEVO,10,2017-11-10T17:00:03.000Z,"""Eminem""|""Walk""|""On""|""Water""|""Aftermath/Shady/Interscope""|""Rap""",17158579,787420,43420,125882,https://i.ytimg.com/vi/n1WpP7iowLc/default.jpg,False,False,False,Eminem's new track Walk on Water ft. Beyoncé is available everywhere: http://shady.sr/WOWEminem \nPlaylist Best of Eminem: https://goo.gl/AquNpo\nSubscribe for more: https://goo.gl/DxCrDV\n\nFor more visit: \nhttp://eminem.com\nhttp://facebook.com/eminem\nhttp://twitter.com/eminem\nhttp://instagram.com/eminem\nhttp://eminem.tumblr.com\nhttp://shadyrecords.com\nhttp://facebook.com/shadyrecords\nhttp://twitter.com/shadyrecords\nhttp://instagram.com/shadyrecords\nhttp://trustshady.tumblr.com\n\nMusic video by Eminem performing Walk On Water. (C) 2017 Aftermath Records\nhttp://vevo.ly/gA7xKt,GB
PUTEiSjKwJU,17.14.11,Goals from Salford City vs Class of 92 and Friends at The Peninsula Stadium!,Salford City Football Club,17,2017-11-13T02:30:38.000Z,"""Salford City FC""|""Salford City""|""Salford""|""Class of 92""|""University of Salford""|""Salford Uni""|""Non League""|""National League""|""National League North""",27833,193,12,37,https://i.ytimg.com/vi/PUTEiSjKwJU/default.jpg,False,False,False,Salford drew 4-4 against the Class of 92 and Friends at the newly opened The Peninsula Stadium!\n\nLike us on Facebook: https://www.facebook.com/SalfordCityFC/ \nFollow us on 

#### Step2: Check Dataset Size

In [0]:
print("Bronze Rows:", bronze_df.count())

Bronze Rows: 158098


#### Step 3: Remove Duplicate Records

In [0]:
silver_df = bronze_df.dropDuplicates(
    ["video_id", "country"]
)

##### Verification:

In [0]:
print("Silver Rows:", silver_df.count())

Silver Rows: 50357


#### Step 4: Handle Null Values

In [0]:
silver_df = silver_df.na.fill({
    "views": 0,
    "likes": 0,
    "dislikes": 0,
    "comment_count": 0
})

#### Step 5: Convert Publish Time

In [0]:
from pyspark.sql.functions import to_timestamp

silver_df = silver_df.withColumn(
    "publish_time",
    to_timestamp("publish_time")
)

##### Verification:

In [0]:
silver_df.select(
    "publish_time"
).show(5, False)

+-------------------+
|publish_time       |
+-------------------+
|2017-11-12 05:18:02|
|2017-11-12 23:20:25|
|2017-11-11 02:41:22|
|2017-11-14 21:00:02|
|2017-11-15 12:07:54|
+-------------------+
only showing top 5 rows


#### Step 6: Extract Time Features

In [0]:
from pyspark.sql.functions import (
    year,
    month,
    dayofmonth,
    hour
)

silver_df = silver_df \
.withColumn(
    "publish_year",
    year("publish_time")
) \
.withColumn(
    "publish_month",
    month("publish_time")
) \
.withColumn(
    "publish_day",
    dayofmonth("publish_time")
) \
.withColumn(
    "publish_hour",
    hour("publish_time")
)

#### Step 7: Create Engagement Rate KPI

In [0]:
from pyspark.sql.functions import col, when

silver_df = silver_df.withColumn(
    "engagement_rate",
    when(
        col("views") > 0,
        (
            col("likes") +
            col("comment_count")
        ) / col("views")
    ).otherwise(0)
)

##### Verification

In [0]:
silver_df.select(
        "video_id",
        "channel_title",
        "views",
        "likes",
        "comment_count",
        "engagement_rate"
    ).show(20, False)

+-----------+---------------------------+-------+------+-------------+---------------------+
|video_id   |channel_title              |views  |likes |comment_count|engagement_rate      |
+-----------+---------------------------+-------+------+-------------+---------------------+
|1UE5Dq1rvUA|Ken Reactz                 |320964 |8069  |717          |0.027373786468264355 |
|1KHCqtTC0EQ|Motion Station             |102766 |294   |39           |0.0032403713290387872|
|GjP_A1Q1sjM|万维TV                     |320294 |600   |370          |0.0030284675953967293|
|cmoknv58jjE|Tanner Braungardt          |1803724|57470 |7483         |0.03601049828022469  |
|dq6G2YWoRqA|orelsan                    |1267473|111445|5994         |0.09265601713014794  |
|UhZ56rcWwRQ|Disney Movie Trailers      |165139 |3497  |557          |0.02454901628325229  |
|CZgHbTfALLo|Philly D                   |113756 |9810  |735          |0.09269840711698724  |
|ON5YYRxcgVM|Fresh Media Records        |133590 |5598  |568          |0.

In [0]:
silver_df.select("engagement_rate").summary().show()

+-------+--------------------+
|summary|     engagement_rate|
+-------+--------------------+
|  count|               50357|
|   mean|0.036733515898103246|
| stddev| 0.03911959459811495|
|    min|                 0.0|
|    25%|0.008378492708982154|
|    50%|0.022736894924184216|
|    75%|0.053227872741100205|
|    max|  0.7742822580932931|
+-------+--------------------+



#### Step 8: Create Like Percentage KPI

In [0]:
silver_df = silver_df.withColumn(
    "like_percentage",
    when(
        col("views") > 0,
        (col("likes") / col("views")) * 100
    ).otherwise(0)
)

#### Step 9: Final Data Quality Check

In [0]:
print("Rows:", silver_df.count())

print(
    "Null Engagement:",
    silver_df.filter(
        col("engagement_rate").isNull()
    ).count()
)

Rows: 50357
Null Engagement: 0


#### Step 10: Preview Silver Dataset

In [0]:
display(
    silver_df.select(
        "video_id",
        "channel_title",
        "views",
        "likes",
        "engagement_rate",
        "country"
    ).limit(10)
)

video_id,channel_title,views,likes,engagement_rate,country
1UE5Dq1rvUA,Ken Reactz,320964,8069,0.027373786468264355,CA
1KHCqtTC0EQ,Motion Station,102766,294,0.0032403713290387872,CA
GjP_A1Q1sjM,万维TV,320294,600,0.0030284675953967293,CA
cmoknv58jjE,Tanner Braungardt,1803724,57470,0.03601049828022469,CA
dq6G2YWoRqA,orelsan,1267473,111445,0.09265601713014794,CA
UhZ56rcWwRQ,Disney Movie Trailers,165139,3497,0.02454901628325229,CA
CZgHbTfALLo,Philly D,113756,9810,0.09269840711698724,CA
ON5YYRxcgVM,Fresh Media Records,133590,5598,0.046156149412381164,CA
_1d9WFw1uKM,Good Mythical Morning,312362,8011,0.027669818992066896,CA
ik_IIjEyZt0,HAR PAL GEO,106965,920,0.010956855046043098,CA


#### Step 11: Save Silver Layer

In [0]:
silver_path = "/Volumes/workspace/default/youtube_data/silver"

silver_df.write \
.format("delta") \
.mode("overwrite") \
.save(silver_path)

##### Verification:

In [0]:
silver_check = spark.read \
.format("delta") \
.load(silver_path)

print("Rows:", silver_check.count())

display(silver_check.limit(10))

Rows: 50357


video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country,publish_year,publish_month,publish_day,publish_hour,engagement_rate,like_percentage
1UE5Dq1rvUA,17.14.11,Taylor Swift Perform Ready For It - SNL,Ken Reactz,24,2017-11-12T05:18:02.000Z,[none],320964,8069,285,717,https://i.ytimg.com/vi/1UE5Dq1rvUA/default.jpg,False,False,False,Thanks for watching please subscribe and subscribe like this video\n\nFollow me on ig @fuego_ken,CA,2017,11,12,5,0.027373786468264355,2.513989107812714
1KHCqtTC0EQ,17.14.11,"Toronto Raptors vs Boston Celtics: November 12, 2017",Motion Station,17,2017-11-12T23:20:25.000Z,"""sp:ty=high""|""sp:dt=2017-11-12T15:30:00-05:00""|""sp:vl=en-US""|""sp:st=basketball""|""sp:li=nba""|""sp:ti:home=BOS""|""sp:ti:away=TOR""",102766,294,39,39,https://i.ytimg.com/vi/1KHCqtTC0EQ/default.jpg,False,False,False,Al Horford scores 21 points as the Boston Celtics beat the Toronto Raptors 95-94,CA,2017,11,12,23,0.0032403713290387872,0.28608683805928026
GjP_A1Q1sjM,17.14.11,川普离京后立马“翻脸” 中国发布“惊人”政策（《万维读报》20171110）,万维TV,22,2017-11-11T02:41:22.000Z,"""creaders""|""万维TV""|""万维读者网""|""万维读报""|""川普""|""习近平""|""双十一""|""马云""|""淘宝""|""天猫""|""沙特""|""王子""|""国王""|""台湾""|""解放军""|""特朗普""",320294,600,254,370,https://i.ytimg.com/vi/GjP_A1Q1sjM/default.jpg,False,False,False,台湾军事弱点大曝光；川普为何如此讨中国喜欢？,CA,2017,11,11,2,0.0030284675953967293,0.18732789249876675
cmoknv58jjE,17.16.11,FLIPPING OVER SUPERCAR! *GONE VERY WRONG*,Tanner Braungardt,22,2017-11-14T21:00:02.000Z,"""tanner""|""tanner braungardt""|""trampoline tricks""|""cliff jumping""|""teaching a backflip""|""flip over car""|""flipping over supercar""|""flip over car gone wrong""|""car flip accident""|""horrible accident""|""near death experience""|""near death fail""|""car fail caught on camera""|""tumbling fails""|""gymnastic fails""|""worst fails compilation""|""epic fail compilation""|""audi r8 crash""|""supercar crash compilation""|""supercar crash""|""luxary car crash""|""redbull""|""parkour fail""|""parkour and freerunning""",1803724,57470,5261,7483,https://i.ytimg.com/vi/cmoknv58jjE/default.jpg,False,False,False,"Soooo that happened...\n\nFLIPPER- https://www.instagram.com/bagels_payne/\n\nhttps://www.youtube.com/user/baileypayne12\n\nCAR/DRIVER/SCARED CHILD- https://www.instagram.com/tannerbraungardt/\n\nJACK (BROTHER)- https://www.instagram.com/jackthepayne/\n\nhttps://www.youtube.com/channel/UCmbTQB6UFr2AbPs0g0FmruA\n\nDOM- https://www.instagram.com/domitrick/\n\nhttps://www.youtube.com/channel/UCS3-mVxKDzUucEYD2gYTnGw/featured\n\nDon't forget to leave a LIKE and SUBSCRIBE - http://bit.ly/subTannerBraungardt - if you enjoyed! Also, SHARE with your friends! \n\nWATCH MORE:\n\nTRAMPOLINE VS - https://www.youtube.com/playlist?list=PLfISKSyDz8gy8BRM1-OBoLKl92z_xfQ6g\nCHALLENGES - https://www.youtube.com/playlist?list=PLfISKSyDz8gxitc78S6EFA7-WpklbHP_B\nLATEST - https://www.youtube.com/playlist?list=PLfISKSyDz8gxRFITdSH-lpVGQ1Pnc0YXl\n\nFOLLOW ME:\nInstagram: https://instagram.com/tannerbraungardt/\nSnapchat: http://snapchat.com/add/tanner_b24\nTwitter: https://twitter.com/braungardtanner\nFacebook: https://facebook.com/tannerbraungardt/\n\nOfficial Merch:\nhttp://www.tbraungardt.com/\n\nBusiness Inquiries: braungardtbiz@gmail.com\n\nFAN MAIL ADDRESS: \nTanner Braungardt\nP.O. Box 98\nAugusta, KS 67010\n\nSkybound's Amazon: http://amzn.to/2bWpQlc\nSkyBound Stratos - 10% Off\nUse Code: TANNER10 http://bit.ly/2cOikfQ\nSkyBoundUSA.com - http://bit.ly/2cr7QSl\n\nOutro music: Zara Larsson - Ain't My Fault (R3hab Remix)\n\nIf you've read all of this I love you\n\n-----------------------------------------------------------------------------------------------",CA,2017,11,14,21,0.03601049828022469,3.18618591314414
dq6G2YWoRqA,17.16.11,OrelSan - Tout va bien [CLIP OFFICIEL],orelsan,10,2017-11-15T12:07:54.000Z,[none],1267473,111445,1663,5994,https://i.ytimg.co